# Lesson 12 - Pagbawas ng Kasaysayan ng Chat gamit ang Agent Scratchpad

Ipinapakita ng notebook na ito kung paano pamahalaan ang konteksto sa mahabang pag-uusap gamit ang Microsoft Agent Framework. Habang lumalaki ang pag-uusap, tumataas ang bilang ng token — hanggang sa lumampas sa konteksto ng modelo. Nilulutas namin ito gamit ang **pattern ng pagbubuod ng konteksto** at isang **agent scratchpad** para sa persistenteng memorya.

## Ano ang Matututunan Mo:
1. **Bakit Mahalaga ang Pamamahala ng Konteksto**: Pag-unawa sa mga limitasyon ng token at konteksto
2. **Context-Aware Agents**: Pagbuo ng mga agent na namamahala ng sariling konteksto ng pag-uusap
3. **Pattern ng Pagbubuod ng Konteksto**: Paggamit ng mga tool para paikliin ang kasaysayan ng pag-uusap
4. **Agent Scratchpad**: Persistenteng memorya na nananatili sa kabila ng pagbawas ng konteksto

## Mga Kinakailangan:
- Azure OpenAI na naka-setup at naka-configure ang mga environment variable
- Pag-unawa sa mga batayang konsepto ng agent mula sa mga nakaraang aralin


## Pagsasaayos


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Bakit Mahalaga ang Pamamahala ng Konteksto

Ang bawat LLM ay may hangganang **context window** — ang pinakamalaking bilang ng mga token na kaya nitong iproseso sa isang kahilingan. Habang nagpapatuloy ang isang multi-turn na pag-uusap:

- **Ang bilang ng token ay tumataas nang linear** sa bawat mensahe ng user at tugon ng assistant.
- **Ang mga prompt token ang nangingibabaw sa gastos** dahil ang buong kasaysayan ay muling ipinapadala sa bawat turno.
- Sa huli, ang pag-uusap ay **lalampas sa context window** at ang modelo ay magpuputol o magbibigay ng error.

### Mga Estratehiya sa Pamamahala ng Konteksto

| Estratehiya | Paano Ito Gumagana | Kapalit |
|---|---|---|
| **Pagpuputol** | Tanggalin ang pinakalumang mga mensahe | Nawawala ang unang konteksto |
| **Pagbubuod** | Pagsamahin ang mga lumang mensahe sa isang buod | May ilang detalye na nawawala, ngunit ang mga pangunahing punto ay nananatili |
| **Scratchpad / Panlabas na Memorya** | Itago ang mga mahalagang impormasyon sa labas ng pag-uusap | Nangangailangan ng tawag sa tool, ngunit nananatili sa kabila ng anumang pagbabawas |

Sa notebook na ito, pinagsasama namin ang **pagbubuod** sa isang **scratchpad tool** upang mapanatili ng ahente ang tuloy-tuloy na daloy kahit na paikliin ang kasaysayan ng pag-uusap.


## Paglikha ng Isang Ahenteng May Kamalayan sa Konteksto


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pagsasagawa ng Isang Mahabang Usapan

Tuklasin natin ang isang pag-uusap na may maraming palitan upang makita kung paano naiipon ang konteksto. Dapat matandaan ng ahente ang mga mahahalagang detalye (mga kagustuhan, badyet, mga petsa ng paglalakbay) sa bawat palitan at ipakita ang tuloy-tuloy na daloy.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pansinin kung paano nananatili ang konteksto ng ahente mula sa mga naunang bahagi ng usapan — alam nito ang tungkol sa Japan, sushi, mga templo, potograpiya, ang badyet na $3000, solo na paglalakbay, at ang panahon ng Abril. Sa isang maikling usapan, mahusay itong gumagana, ngunit habang lumalaki ang usapan, nagiging magastos na muling ipadala ang buong kasaysayan.

Ipagpatuloy natin ang usapan sa mas maraming bahagi upang makita ang pag-ipon ng konteksto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

Habang lumalaki ang usapan, maaari nating gamitin ang isang **tool sa pagbubuod** upang paikliin ang naipong konteksto sa isang compact na porma. Tinatawag ng ahente ang tool na ito upang itala ang mga pangunahing nais upang kahit tanggalin ang mga lumang mensahe, mapanatili ang mahahalagang impormasyon.

Ang pattern na ito ay pundasyon para sa mas sopistikadong pagbabawas ng kasaysayan:
1. Tinutukoy ng ahente ang mga pangunahing katotohanan mula sa usapan
2. Tinatawag nito ang tool sa pagbubuod upang itala ang mga ito
3. Maaaring ligtas na tanggalin ang mga lumang mensahe dahil nahuhuli ng buod ang mga mahalaga

Sa ibaba ay tinutukoy namin ang isang `summarize_preferences` tool na maaaring tawagin ng ahente upang itala ang compact na buod ng mga natutunan nito.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Buod

Sa araling ito natutunan mo kung paano pamahalaan ang konteksto sa mga matagalang pag-uusap ng ahente gamit ang Microsoft Agent Framework:

### Mga Pangunahing Konsepto
- **May hangganan ang mga bintana ng konteksto** — bawat token sa kasaysayan ng pag-uusap ay may halaga at binibilang patungo sa limitasyon.
- **Mga kasangkapang pangbuod** ay nagpapahintulot sa ahente na paliitin ang naipong konteksto sa mas compact na mga buod, nagpapababa sa paggamit ng token habang pinapanatili ang mahalagang impormasyon.
- **Agent scratchpads** ay nagbibigay ng matibay na panlabas na memorya na nananatili kahit pa bawasan ang pag-uusap.

### Ang Iyong Naitayo
- Isang **ahenteng may kamalayan sa konteksto** na nagpapanatili ng tuloy-tuloy na daloy sa mga multi-turn na pag-uusap
- Isang **kasangkapan sa pagbubuod** (`summarize_preferences`) na nagrerekord ng mahahalagang detalye ng user sa compact na anyo
- Isang **multi-turn na pag-uusap** na nagpapakita ng pagpapanatili ng konteksto at paghawak sa mga pagbabago

### Mga Aplikasyon sa Tunay na Mundo
- **Customer Service Bots**: Naaalala ang mga kagustuhan sa matagal na mga sesyon ng suporta
- **Personal Assistants**: Sinusubaybayan ang mga kasalukuyang proyekto nang hindi kailangang ulitin ang konteksto
- **Educational Tutors**: Pinapanatili ang progreso ng estudyante sa maraming interaksyon

### Susunod na Mga Hakbang
- Magpatupad ng kumpletong scratchpad tool na may file-based persistence
- Magdagdag ng awtomatikong pagbawas ng kasaysayan pagkatapos ng pagbubuod
- Pagsamahin sa mga vector databases para sa semantic memory search
- Bumuo ng mga ahente na maaaring ipagpatuloy ang mga pag-uusap ilang araw pagkatapos na may buong konteksto


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pahayag ng Pagsasalin**:
Ang dokumentong ito ay isinalin gamit ang AI na serbisyo ng pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsisikap kaming maging tumpak, mangyaring tandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa sariling wika nito ang dapat ituring na pangunahing sanggunian. Para sa mga mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang mga hindi pagkakaunawaan o maling interpretasyon na maaaring mabuo mula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
